Hypothesis testing & PRC of simple de novo simulation.

In [1]:
#imports
import scMPRAforge as scm
import pandas as pd
import numpy as np
from dask.distributed import Client, LocalCluster

2025-09-16 10:35:23.805316: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-16 10:35:23.809528: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /vast/palmer/apps/avx2/software/Code-Server/4.17.0/lib:/vast/palmer/apps/avx2/software/gettext/0.22.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libiconv/1.17-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/ncurses/6.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/XZ/5.4.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/expat/2.6.2-GCCcore-13.3.0/lib:/vast/palmer/apps/av

In [2]:
#autoreload
%load_ext autoreload
%autoreload 2

In [3]:
#set up cluster

#cluster=LocalCluster(memory_limit='4GB',n_workers=7,threads_per_worker=4,)
#client=Client(cluster)



from dask_jobqueue import SLURMCluster
from dask.distributed import Client

cluster=SLURMCluster(
    cores=4,#cores per slurm job
    memory="128G",#memory per slurm job
    processes=8,#dask workers
    job_extra_directives=["-p ycga", 
        f"--job-name=simclust_worker",
        f"--time=8:00:00",
        f"--output=slave_%j.out"]
)

cluster.scale(jobs=6)

client = Client(cluster,
        timeout=f"{5*60}s",   # Client <-> scheduler timeout 
        heartbeat_interval="20s"  # Worker heartbeat interval
    )


In [4]:
#load sim & hypo test
data_root="/gpfs/gibbs/pi/reilly/tabula_data"

hypothesis_set=scm.HypothesisSet.from_tsv(f"{data_root}/simulated/activity_de_novo_hypotheses.tsv")

In [5]:
de_novo_sim=None
first_time=True
if first_time:
    de_novo_sim=scm.de_novo_simulation.load(client,data_root,"simulated/activity_de_novo")
    de_novo_sim._test_all_replicates(client=client,
        hypothesis_set=hypothesis_set,
        test="wald")
    de_novo_sim.save(data_root,"simulated/activity_de_novo_and_results")
else:
    de_novo_sim=scm.de_novo_simulation.load(client,data_root,"simulated/activity_de_novo_and_results")

KeyboardInterrupt: 

In [ ]:
de_novo_sim.results

In [ ]:
def _gt_comp_helper():
    pass

#public method of results object
def compare_to_ground_truth(self,gt):
    """
    Takes a ground truth dataframe.
    Returns a single row frame with lots of useful information about 
    how well the test recapitulated the ground truth. 
    """
    pass

In [ ]:
#public method of de_novo_batch
def summarize_performance(self,hypothesis_set,test):
    """
    Takes a hypothesis set object and a particular test type string and produces a dataframe 
    summarzing performance for all simulated replicates, and saves to `self.performances`.
    Then aggregates all metrics to single values & saves to `self.performance_all`
    This function is a collector and will hang!
    """
    pass
    #Calls _test_all_replicates, then 
    #calls compare_to_ground_truth on each. 
    #aggregates the resuling DFs and 

In [ ]:
#public method of de_novo_batch
def performance_plot(self,plot_type,*args,**kwargs):
    """
    Plots the performance of the executed hypothesis tests.
    
    Requires that you call summarize_performance first. 
    Draws from `self.performance_all`

    args and kwargs are passed to the plotting function to allow 
    the user to modify the appearance of the plot as desired. 
    
    plot type can be one of....
    (for each option show which plotting function the args will be passed to)

    
    """
    pass

In [ ]:
#merge in `comparison` ground truth
merged=results.merge(de_novo_sim.ground_truth,
    left_on=["comparison_CRE","comparison_cell_type"],
    right_on=["cre_id","cell_type"]
)
merged=merged.drop(columns=["cre_id","cell_type"])
merged=merged.rename({"true_mean":"comparison_truth"},axis=1)

#merge in `reference` ground truth
merged=merged.merge(de_novo_sim.ground_truth,
    left_on=["reference_CRE","reference_cell_type"],
    right_on=["cre_id","cell_type"]
)
merged=merged.drop(columns=["cre_id","cell_type"])
merged=merged.rename({"true_mean":"reference_truth"},axis=1)

#ground truth effect size
merged["gt_effect_size"]=merged["comparison_truth"]/merged["reference_truth"]
#ground truth null hypothesis that the CREs are the same : true or false?
merged["gt_null"]=abs(merged["gt_effect_size"]-1)<1e-8

merged["reject_null"]=merged["bh_p"]<0.05

simple_bh_results=pd.crosstab(merged["gt_null"],merged["reject_null"])

#now onto PRC

import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score, roc_curve, roc_auc_score

y_true = (~merged["gt_null"]).astype(int).to_numpy()
p = merged["p_value"].to_numpy()
epsilon = np.finfo(float).tiny
scores = -np.log10(p + epsilon)

# 3) PR curve + AUPRC
prec, rec, _ = precision_recall_curve(y_true, scores)
auprc = average_precision_score(y_true, scores)

# 4) ROC (optional)
fpr, tpr, _ = roc_curve(y_true, scores)
auroc = roc_auc_score(y_true, scores)

# 5) Plot PRC with baseline
pos_rate = y_true.mean()  # prevalence; PRC baseline
plt.figure(figsize=(5,4))
plt.plot(rec, prec, lw=2)
plt.hlines(pos_rate, 0, 1, linestyles="--", label=f"Baseline = {pos_rate:.3f}")
plt.xlim(0, 1); plt.ylim(0, 1)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"PR Curve (AUPRC = {auprc:.3f})")
plt.legend()
plt.tight_layout()
plt.show()

# (Optional) Plot ROC
plt.figure(figsize=(5,4))
plt.plot(fpr, tpr, lw=2, label=f"AUROC = {auroc:.3f}")
plt.plot([0,1],[0,1], "--", color="gray")
plt.xlim(0,1); plt.ylim(0,1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.tight_layout()
plt.show()


In [7]:
cluster.close()